# 노지 작물 해충 진단 (서브셋) - Qwen3.5-9B LoRA 파인튜닝

**대상**: 썩덩나무노린재 + 정상 (2클래스)  
**데이터셋**: `data/` (로컬)  
**모델**: Qwen3.5-9B (bf16 LoRA)  
**환경**: 32GB+ VRAM (A5000/A6000)

---

## 1. 패키지 설치

In [ ]:
!pip install --upgrade pip
!pip install --upgrade typing_extensions
!pip install unsloth
!pip install "transformers>=5.2"
!pip install trl==0.22.2 datasets Pillow accelerate scikit-learn huggingface_hub


In [ ]:
# 설치 확인
from unsloth import FastVisionModel
print("Unsloth OK")

## 2. 데이터셋 경로 설정

In [ ]:
import os

DATA_DIR = "/workspace/data"

assert os.path.exists(os.path.join(DATA_DIR, "train.jsonl")), \
    f"데이터셋이 없습니다: {DATA_DIR}/train.jsonl"
print(f"데이터셋 경로: {DATA_DIR}")

## 3. 이미지 전처리 (크롭 → 디스크 저장)

해충 이미지는 바운딩 박스 기반 크롭 (tight 50% / context 50%)  
정상 이미지는 원본 유지  
결과를 디스크에 저장하고 새로운 JSONL 생성 → **RAM을 거의 안 씀**

In [ ]:
import json
import os
import random
from PIL import Image

Image.MAX_IMAGE_PIXELS = None

USER_PROMPT = "이 식물 사진에서 해충을 판별해줘. 해충이 없으면 '정상', 있으면 종류를 말해줘."

SYSTEM_MSG = "당신은 작물 해충 식별 전문가입니다. 부가 설명 없이 이름만 출력하세요."

BBOX_GROW_STAGE = 33  # 생육 단계 33 = 해충 바운딩 박스 라벨


def crop_to_bbox(img, bbox, padding_ratio=0.0):
    """바운딩 박스 기준으로 크롭"""
    xtl, ytl = bbox["xtl"], bbox["ytl"]
    xbr, ybr = bbox["xbr"], bbox["ybr"]
    bw, bh = xbr - xtl, ybr - ytl
    pad_x, pad_y = int(bw * padding_ratio), int(bh * padding_ratio)
    x1 = max(0, xtl - pad_x)
    y1 = max(0, ytl - pad_y)
    x2 = min(img.width, xbr + pad_x)
    y2 = min(img.height, ybr + pad_y)
    return img.crop((x1, y1, x2, y2))


def find_label_json(split, class_name, img_filename):
    """JSON 라벨에서 바운딩 박스 반환"""
    json_path = os.path.join(DATA_DIR, split, class_name, img_filename + ".json")
    if not os.path.exists(json_path):
        return None
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    for obj in data["annotations"]["object"]:
        if obj["grow"] == BBOX_GROW_STAGE and obj.get("points"):
            return obj["points"][0]
    return None


def preprocess_split(split="train"):
    """원본 이미지를 크롭하여 디스크에 저장하고 새 JSONL 생성"""
    jsonl_path = os.path.join(DATA_DIR, f"{split}.jsonl")
    out_dir = os.path.join(DATA_DIR, f"{split}_cropped")
    out_jsonl = os.path.join(DATA_DIR, f"{split}_cropped.jsonl")

    # 이미 전처리 완료 확인
    if os.path.exists(out_jsonl):
        with open(out_jsonl, "r") as f:
            count = sum(1 for _ in f)
        print(f"  [{split}] 이미 전처리 완료: {count}건")
        return

    # 총 라인 수
    with open(jsonl_path, "r", encoding="utf-8") as f:
        total = sum(1 for _ in f)

    out_file = open(out_jsonl, "w", encoding="utf-8")
    count = 0

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if (i + 1) % 500 == 0 or (i + 1) == total:
                print(f"\r  [{split}] {i + 1}/{total} ({(i + 1) * 100 // total}%)", end="", flush=True)

            record = json.loads(line)
            messages = record["messages"]
            label = messages[-1]["content"][0]["text"]

            # 이미지 경로 추출
            img_rel_path = None
            for msg in messages:
                for content in msg["content"]:
                    if content["type"] == "image" and "image" in content:
                        img_rel_path = content["image"].replace("\\", "/")
                        break

            if img_rel_path is None:
                continue

            parts = img_rel_path.split("/")
            class_name = parts[1]
            img_filename = parts[2]

            # 출력 디렉토리 생성
            class_dir = os.path.join(out_dir, class_name)
            os.makedirs(class_dir, exist_ok=True)

            # 출력 파일명
            base_name = os.path.splitext(img_filename)[0]
            out_filename = f"{base_name}.jpg"
            out_path = os.path.join(class_dir, out_filename)
            out_rel_path = f"{split}_cropped/{class_name}/{out_filename}"

            # 이미지 처리
            img_path = os.path.join(DATA_DIR, img_rel_path)
            img = Image.open(img_path).convert("RGB")

            if label == "정상":
                result = img
            else:
                bbox = find_label_json(split, class_name, img_filename)
                if bbox:
                    r = random.random()
                    if r < 0.5:
                        result = img
                    elif r < 0.75:
                        result = crop_to_bbox(img, bbox, padding_ratio=0.0)
                    else:
                        result = crop_to_bbox(img, bbox, padding_ratio=0.5)
                else:
                    result = img

            # 디스크에 저장
            result.save(out_path, "JPEG", quality=95)
            if result is not img:
                result.close()
            img.close()

            # 새 JSONL 레코드 작성
            new_record = {
                "messages": [
                    {"role": "system", "content": [
                        {"type": "text", "text": SYSTEM_MSG}
                    ]},
                    {"role": "user", "content": [
                        {"type": "image", "image": out_rel_path},
                        {"type": "text", "text": random.choice(PROMPTS)},
                    ]},
                    {"role": "assistant", "content": [
                        {"type": "text", "text": label}
                    ]},
                ]
            }
            out_file.write(json.dumps(new_record, ensure_ascii=False) + "\n")
            count += 1

    out_file.close()
    print(f"\n  [{split}] 완료: {count}건 → {out_dir}")


random.seed(42)
print("=== 이미지 전처리 (크롭 → 디스크 저장) ===")
preprocess_split("train")
preprocess_split("val")
print("전처리 완료!")

## 4. 데이터 로딩 (경로 기반 — RAM 절약)

In [ ]:
import json
import os
import random
from PIL import Image

def load_dataset_from_cropped_jsonl(split="train"):
    """전처리된 JSONL을 읽어 이미지 경로만 참조하는 대화 형식 생성 (RAM 절약)"""
    jsonl_path = os.path.join(DATA_DIR, f"{split}_cropped.jsonl")
    dataset = []

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            messages = record["messages"]

            # 이미지 경로를 절대경로로 변환
            for msg in messages:
                for content in msg["content"]:
                    if content["type"] == "image" and "image" in content:
                        content["image"] = os.path.join(DATA_DIR, content["image"])

            dataset.append(record)

    random.shuffle(dataset)
    return dataset


random.seed(42)
train_dataset = load_dataset_from_cropped_jsonl("train")
val_dataset = load_dataset_from_cropped_jsonl("val")
print(f"Train: {len(train_dataset)}건, Val: {len(val_dataset)}건")

## 5. 모델 로딩

In [ ]:
import os
os.environ["HF_HOME"] = "/workspace/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/workspace/hf_cache"

# 기존 캐시 삭제해서 container disk 확보
!rm -rf /root/.cache/huggingface
!mkdir -p /workspace/hf_cache
print("캐시 경로 변경 완료")


In [ ]:
import torch
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-9B",
    load_in_4bit=False,
    use_gradient_checkpointing="unsloth",
)

## 6. LoRA 설정

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

## 7. 학습 설정 및 실행

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        per_device_train_batch_size=6,
        gradient_accumulation_steps=2,
        warmup_steps=50,
        num_train_epochs=3,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=10,
        save_strategy="epoch",
        eval_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="pest-detector-qwen3.5",
        report_to="wandb",
        run_name="pest-subset-qwen3.5",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=2048,
        dataloader_num_workers=8,
        dataloader_pin_memory=True,
    ),
)

print("학습 시작!")
trainer.train()
print("학습 완료!")

## 8. 모델 저장

In [ ]:
LORA_DIR = "pest-detector-lora"

model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"LoRA 어댑터 저장 완료: {LORA_DIR}")

## 9. 추론 테스트

In [ ]:
from unsloth import FastVisionModel
from PIL import Image
from transformers import TextStreamer

# LoRA 어댑터 로딩
model, tokenizer = FastVisionModel.from_pretrained(
    "pest-detector-lora",
    load_in_4bit=False,
)
FastVisionModel.for_inference(model)

In [ ]:
# 테스트 이미지 경로를 지정하세요
TEST_IMAGE = "/workspace/data/val/벼룩잎벌레/V006_78_2_12_00_07_33_0_8547q_20201129_8243.JPG"  # 예시

image = Image.open(TEST_IMAGE).convert("RGB")

messages = [
    {"role": "system", "content": [
        {"type": "text", "text": SYSTEM_MSG}
    ]},
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": "이 사진에 있는 해충의 이름을 알려주세요."},
    ]},
]

input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens=False,
    return_tensors="pt",
).to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print("예측 결과: ", end="")
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=20,
    use_cache=True,
)

## 10. HuggingFace Hub에 업로드

In [ ]:
import os
from huggingface_hub import login

login(token=os.environ["HF_TOKEN"])

In [ ]:
import os

# 모델 + 토크나이저 함께 업로드
model.push_to_hub(
    "YOUR_REPO_NAME",
    tokenizer,
    token=os.environ["HF_TOKEN"],
)

print("업로드 완료!")

In [ ]:
!rm -rf /root/.cache
!rm -rf /tmp/*
!df -h /
